In [33]:
# DEPENDENCIAS
!pip install tslearn

In [34]:
# LIBRERIAS
import zipfile
import os
import shutil
import pandas as pd
import numpy as np
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import matplotlib.pyplot as plt


In [35]:
# CARGAR Y EXTRAER ARCHIVOS ZIP
# Crear carpeta "Original" y "Generado"
os.makedirs("Original", exist_ok=True)
os.makedirs("Generado", exist_ok=True)

zip_path_1 = "/content/Categorias.zip"         # Ruta del ZIP
extract_path_1 = "Original"                    # Carpeta donde se descomprime
zip_path_2 = "/content/Resultados_Modelo.zip"  # Ruta del ZIP
extract_path_2 = "Generado"                    # Carpeta donde se descomprime

# Descomprimir 1
with zipfile.ZipFile(zip_path_1, 'r') as zip_ref:
    zip_ref.extractall(extract_path_1)
print(f"Archivos extraídos en: {extract_path_1}")
with zipfile.ZipFile(zip_path_2, 'r') as zip_ref:
    zip_ref.extractall(extract_path_2)
print(f"Archivos extraídos en: {extract_path_2}")

Archivos extraídos en: Original
Archivos extraídos en: Generado


In [36]:
# FUNCION PARA APILAR LOS LAS FEATURES A PARTIR DE UN MES
def clusters_temporales(path, indice_var):
    # Carpeta donde están tus CSVs
    folder_path = path
    csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')])

    # Lista para guardar la primera columna de cada CSV
    columnas = []

    for file in csv_files:
        df = pd.read_csv(os.path.join(folder_path, file))

        # Tomar la primera columna
        col = df.iloc[:, indice_var]  # columna índice x
        col.name = f"{file}_col1"  # renombrar columna para identificar su origen

        columnas.append(col)

    # Concatenar todas las columnas horizontalmente
    nuevo_df = pd.concat(columnas, axis=1)

    return nuevo_df

In [37]:
# Funcion para graficar clusters
def graficar_clusters(nuevo_df,n_clusters, titulo_graf, path):
  # Serie temporal (24 pasos)
  series = nuevo_df.T.values  # shape (n_series, n_timesteps)
  series = series[:, :, np.newaxis]  # tslearn espera shape (n_series, timesteps, dim=1)

  # Clustering usando KMeans para series temporales
  n_clusters = n_clusters  # Ajustar cantidad de clusters
  model = TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", max_iter=10, random_state=42)
  labels = model.fit_predict(series)

  # Graficar resultados
  plt.figure(figsize=(12,6))
  for cluster in range(n_clusters):
      #cluster_series = series_scaled[labels == cluster]
      cluster_series = series[labels == cluster]
      for s in cluster_series:
          plt.plot(s.ravel(), color='gray', alpha=0.3)
      plt.plot(cluster_series.mean(axis=0).ravel(), label=f'Cluster {cluster}', linewidth=3)

  plt.xlabel('Paso temporal (hora)')
  plt.ylabel('Valor')
  plt.title(titulo_graf)
  plt.legend()
  plt.show

  # Crear directorio si no existe
  os.makedirs(path, exist_ok=True)

  # Generar nombre de archivo a partir del título (sin espacios)
  nombre_archivo = titulo_graf.replace(" ", "_") + ".png"
  ruta = os.path.join(path, nombre_archivo)

  # Guardar figura
  plt.savefig(ruta, dpi=300, bbox_inches='tight')
  plt.close()  # cerrar la figura para liberar memoria

  print(f"Imagen guardada en: {ruta}")
  return cluster_series.mean(axis=0).ravel()


In [38]:
def graficar_comparativa(vector_cluster_ene_gen, vector_cluster_ene_org, path, titulo_graf,mse):
  # Crear eje x de 0 a 23
  x = np.arange(24)

  # Graficar
  plt.figure(figsize=(10,5))
  plt.plot(x, vector_cluster_ene_gen, marker='o', label="Cluster Generado", linewidth=2)
  plt.plot(x, vector_cluster_ene_org, marker='s', label="Cluster Original", linewidth=2)
  plt.text(0.05, 0.95, f"MSE = {mse}", transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
  plt.xlabel("Tiempo")
  plt.ylabel("Valor")
  titulo_base = "Cluster Generado vs Cluster Original"
  plt.title(f"{titulo_base} - {titulo_graf}")
  plt.legend()
  plt.show

  # Crear directorio si no existe
  os.makedirs(path, exist_ok=True)

  # Generar nombre de archivo a partir del título (sin espacios)
  nombre_archivo = titulo_graf.replace(" ", "_") + ".png"
  ruta = os.path.join(path, nombre_archivo)

  # Guardar figura
  plt.savefig(ruta, dpi=300, bbox_inches='tight')
  plt.close()  # cerrar la figura para liberar memoria

  print(f"Imagen guardada en: {ruta}")


In [39]:
def guardar_datos_graficas(path_datos_original, path_datos_generados, feature, n_clusters, titulo_original, titulo_gen, titulo_comparativa, path_guardado):
  dfaux_1 = clusters_temporales(path_datos_original, feature)                                        # Apilar graficas de la variable '0' original
  vector_cluster_ene_org = graficar_clusters(dfaux_1, n_clusters, titulo_original, path_guardado)    # Generar cluster y graficas de series temporales
  dfaux_2 = clusters_temporales(path_datos_generados, feature)                                       # Apilar graficas de la variable '0' generado
  vector_cluster_ene_gen = graficar_clusters(dfaux_2, n_clusters, titulo_gen, path_guardado)         # Generar cluster y graficas de series temporales
  errores = (vector_cluster_ene_org - vector_cluster_ene_gen) ** 2                                   # Error punto a punto
  mse = np.mean(errores)                                                                             # MSE total
  df_aux_3 = pd.DataFrame({
      "Vector_Original": vector_cluster_ene_org,
      "Vector_Generado": vector_cluster_ene_gen,
      "Errores": errores
  })
  nombre_archivo = titulo_comparativa.replace(" ", "_") + ".xlsx"
  ruta = os.path.join(path_guardado, nombre_archivo)
  df_aux_3.to_excel(ruta, index=False)                                                                # Guardar en Excel
  graficar_comparativa(vector_cluster_ene_gen, vector_cluster_ene_org, path_guardado, titulo_comparativa, mse)

In [40]:
#import shutil

# Eliminar carpeta "Categorias" si existe
#shutil.rmtree("/content/Graficos_resultados", ignore_errors=True)

In [41]:
# Diccionario con nombres e índices Enero 2020
variables_enero_2020 = {
    "Enero 2020 PV": 0,
    "Enero 2020 Wind": 1,
    "Enero 2020 Consumption": 2,
    "Enero 2020 Spot Market Price": 3,
    "Enero 2020 Precip 1h Mm": 4,
    "Enero 2020 Precip Type Idx": 5,
    "Enero 2020 Prob Precip 1h P": 6,
    "Enero 2020 Clear Sky Rad W": 7,
    "Enero 2020 Clear Sky Energy 1h J": 8,
    "Enero 2020 Diffuse Rad W": 9,
    "Enero 2020 Diffuse Rad 1h Wh": 10,
    "Enero 2020 Direct Rad W": 11,
    "Enero 2020 Direct Rad 1h Wh": 12,
    "Enero 2020 Global Rad W": 13,
    "Enero 2020 Global Rad 1h Wh": 14,
    "Enero 2020 Sunshine Duration 1h Min": 15,
    "Enero 2020 Sun Azimuth D": 16,
    "Enero 2020 Sun Elevation D": 17,
    "Enero 2020 Low Cloud Cover P": 18,
    "Enero 2020 Medium Cloud Cover P": 19,
    "Enero 2020 High Cloud Cover P": 20,
    "Enero 2020 Total Cloud Cover P": 21,
    "Enero 2020 Effective Cloud Cover P": 22,
    "Enero 2020 Temp": 23,
    "Enero 2020 Relative Humidity 2m P": 24,
    "Enero 2020 Dew Point 2m C": 25,
    "Enero 2020 Wind Speed 2m Ms": 26,
    "Enero 2020 Wind Dir 2m D": 27,
    "Enero 2020 T 10m C": 28,
    "Enero 2020 Relative Humidity 10m P": 29,
    "Enero 2020 Dew Point 10m C": 30,
    "Enero 2020 Wind Speed 10m Ms": 31,
    "Enero 2020 Wind Dir 10m D": 32,
    "Enero 2020 T 50m C": 33,
    "Enero 2020 Relative Humidity 50m P": 34,
    "Enero 2020 Dew Point 50m C": 35,
    "Enero 2020 Wind Speed 50m Ms": 36,
    "Enero 2020 Wind Dir 50m D": 37,
    "Enero 2020 T 100m C": 38,
    "Enero 2020 Relative Humidity 100m P": 39,
    "Enero 2020 Dew Point 100m C": 40,
    "Enero 2020 Wind Speed 100m Ms": 41,
    "Enero 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_enero_2020.items():
    guardar_datos_graficas(
        '/content/Original/enero_2020',
        '/content/Generado/gen_enero_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Enero_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Enero_2020/0_Clusters_Original_Enero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/0_Clusters_Generado_Enero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/0_Comparativa_Enero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/1_Clusters_Original_Enero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/1_Clusters_Generado_Enero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/1_Comparativa_Enero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/2_Clusters_Original_Enero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/2_Clusters_Generado_Enero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/2_Comparativa_Enero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2020/3_Clusters_Original_Enero_2020_Spot_Market_

In [42]:
# Diccionario con nombres e índices Febrero 2020
variables_febrero_2020 = {
    "Febrero 2020 PV": 0,
    "Febrero 2020 Wind": 1,
    "Febrero 2020 Consumption": 2,
    "Febrero 2020 Spot Market Price": 3,
    "Febrero 2020 Precip 1h Mm": 4,
    "Febrero 2020 Precip Type Idx": 5,
    "Febrero 2020 Prob Precip 1h P": 6,
    "Febrero 2020 Clear Sky Rad W": 7,
    "Febrero 2020 Clear Sky Energy 1h J": 8,
    "Febrero 2020 Diffuse Rad W": 9,
    "Febrero 2020 Diffuse Rad 1h Wh": 10,
    "Febrero 2020 Direct Rad W": 11,
    "Febrero 2020 Direct Rad 1h Wh": 12,
    "Febrero 2020 Global Rad W": 13,
    "Febrero 2020 Global Rad 1h Wh": 14,
    "Febrero 2020 Sunshine Duration 1h Min": 15,
    "Febrero 2020 Sun Azimuth D": 16,
    "Febrero 2020 Sun Elevation D": 17,
    "Febrero 2020 Low Cloud Cover P": 18,
    "Febrero 2020 Medium Cloud Cover P": 19,
    "Febrero 2020 High Cloud Cover P": 20,
    "Febrero 2020 Total Cloud Cover P": 21,
    "Febrero 2020 Effective Cloud Cover P": 22,
    "Febrero 2020 Temp": 23,
    "Febrero 2020 Relative Humidity 2m P": 24,
    "Febrero 2020 Dew Point 2m C": 25,
    "Febrero 2020 Wind Speed 2m Ms": 26,
    "Febrero 2020 Wind Dir 2m D": 27,
    "Febrero 2020 T 10m C": 28,
    "Febrero 2020 Relative Humidity 10m P": 29,
    "Febrero 2020 Dew Point 10m C": 30,
    "Febrero 2020 Wind Speed 10m Ms": 31,
    "Febrero 2020 Wind Dir 10m D": 32,
    "Febrero 2020 T 50m C": 33,
    "Febrero 2020 Relative Humidity 50m P": 34,
    "Febrero 2020 Dew Point 50m C": 35,
    "Febrero 2020 Wind Speed 50m Ms": 36,
    "Febrero 2020 Wind Dir 50m D": 37,
    "Febrero 2020 T 100m C": 38,
    "Febrero 2020 Relative Humidity 100m P": 39,
    "Febrero 2020 Dew Point 100m C": 40,
    "Febrero 2020 Wind Speed 100m Ms": 41,
    "Febrero 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_febrero_2020.items():
    guardar_datos_graficas(
        '/content/Original/febrero_2020',
        '/content/Generado/gen_febrero_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Febrero_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Febrero_2020/0_Clusters_Original_Febrero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/0_Clusters_Generado_Febrero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/0_Comparativa_Febrero_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/1_Clusters_Original_Febrero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/1_Clusters_Generado_Febrero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/1_Comparativa_Febrero_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/2_Clusters_Original_Febrero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/2_Clusters_Generado_Febrero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/2_Comparativa_Febrero_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Febrero_2020/3_Clu

In [43]:
# Diccionario con nombres e índices Marzo 2020
variables_marzo_2020 = {
    "Marzo 2020 PV": 0,
    "Marzo 2020 Wind": 1,
    "Marzo 2020 Consumption": 2,
    "Marzo 2020 Spot Market Price": 3,
    "Marzo 2020 Precip 1h Mm": 4,
    "Marzo 2020 Precip Type Idx": 5,
    "Marzo 2020 Prob Precip 1h P": 6,
    "Marzo 2020 Clear Sky Rad W": 7,
    "Marzo 2020 Clear Sky Energy 1h J": 8,
    "Marzo 2020 Diffuse Rad W": 9,
    "Marzo 2020 Diffuse Rad 1h Wh": 10,
    "Marzo 2020 Direct Rad W": 11,
    "Marzo 2020 Direct Rad 1h Wh": 12,
    "Marzo 2020 Global Rad W": 13,
    "Marzo 2020 Global Rad 1h Wh": 14,
    "Marzo 2020 Sunshine Duration 1h Min": 15,
    "Marzo 2020 Sun Azimuth D": 16,
    "Marzo 2020 Sun Elevation D": 17,
    "Marzo 2020 Low Cloud Cover P": 18,
    "Marzo 2020 Medium Cloud Cover P": 19,
    "Marzo 2020 High Cloud Cover P": 20,
    "Marzo 2020 Total Cloud Cover P": 21,
    "Marzo 2020 Effective Cloud Cover P": 22,
    "Marzo 2020 Temp": 23,
    "Marzo 2020 Relative Humidity 2m P": 24,
    "Marzo 2020 Dew Point 2m C": 25,
    "Marzo 2020 Wind Speed 2m Ms": 26,
    "Marzo 2020 Wind Dir 2m D": 27,
    "Marzo 2020 T 10m C": 28,
    "Marzo 2020 Relative Humidity 10m P": 29,
    "Marzo 2020 Dew Point 10m C": 30,
    "Marzo 2020 Wind Speed 10m Ms": 31,
    "Marzo 2020 Wind Dir 10m D": 32,
    "Marzo 2020 T 50m C": 33,
    "Marzo 2020 Relative Humidity 50m P": 34,
    "Marzo 2020 Dew Point 50m C": 35,
    "Marzo 2020 Wind Speed 50m Ms": 36,
    "Marzo 2020 Wind Dir 50m D": 37,
    "Marzo 2020 T 100m C": 38,
    "Marzo 2020 Relative Humidity 100m P": 39,
    "Marzo 2020 Dew Point 100m C": 40,
    "Marzo 2020 Wind Speed 100m Ms": 41,
    "Marzo 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_marzo_2020.items():
    guardar_datos_graficas(
        '/content/Original/marzo_2020',
        '/content/Generado/gen_marzo_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Marzo_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Marzo_2020/0_Clusters_Original_Marzo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/0_Clusters_Generado_Marzo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/0_Comparativa_Marzo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/1_Clusters_Original_Marzo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/1_Clusters_Generado_Marzo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/1_Comparativa_Marzo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/2_Clusters_Original_Marzo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/2_Clusters_Generado_Marzo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/2_Comparativa_Marzo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Marzo_2020/3_Clusters_Original_Marzo_2020_Spot_Market_

In [44]:
# Diccionario con nombres e índices Abril 2020
variables_abril_2020 = {
    "Abril 2020 PV": 0,
    "Abril 2020 Wind": 1,
    "Abril 2020 Consumption": 2,
    "Abril 2020 Spot Market Price": 3,
    "Abril 2020 Precip 1h Mm": 4,
    "Abril 2020 Precip Type Idx": 5,
    "Abril 2020 Prob Precip 1h P": 6,
    "Abril 2020 Clear Sky Rad W": 7,
    "Abril 2020 Clear Sky Energy 1h J": 8,
    "Abril 2020 Diffuse Rad W": 9,
    "Abril 2020 Diffuse Rad 1h Wh": 10,
    "Abril 2020 Direct Rad W": 11,
    "Abril 2020 Direct Rad 1h Wh": 12,
    "Abril 2020 Global Rad W": 13,
    "Abril 2020 Global Rad 1h Wh": 14,
    "Abril 2020 Sunshine Duration 1h Min": 15,
    "Abril 2020 Sun Azimuth D": 16,
    "Abril 2020 Sun Elevation D": 17,
    "Abril 2020 Low Cloud Cover P": 18,
    "Abril 2020 Medium Cloud Cover P": 19,
    "Abril 2020 High Cloud Cover P": 20,
    "Abril 2020 Total Cloud Cover P": 21,
    "Abril 2020 Effective Cloud Cover P": 22,
    "Abril 2020 Temp": 23,
    "Abril 2020 Relative Humidity 2m P": 24,
    "Abril 2020 Dew Point 2m C": 25,
    "Abril 2020 Wind Speed 2m Ms": 26,
    "Abril 2020 Wind Dir 2m D": 27,
    "Abril 2020 T 10m C": 28,
    "Abril 2020 Relative Humidity 10m P": 29,
    "Abril 2020 Dew Point 10m C": 30,
    "Abril 2020 Wind Speed 10m Ms": 31,
    "Abril 2020 Wind Dir 10m D": 32,
    "Abril 2020 T 50m C": 33,
    "Abril 2020 Relative Humidity 50m P": 34,
    "Abril 2020 Dew Point 50m C": 35,
    "Abril 2020 Wind Speed 50m Ms": 36,
    "Abril 2020 Wind Dir 50m D": 37,
    "Abril 2020 T 100m C": 38,
    "Abril 2020 Relative Humidity 100m P": 39,
    "Abril 2020 Dew Point 100m C": 40,
    "Abril 2020 Wind Speed 100m Ms": 41,
    "Abril 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_abril_2020.items():
    guardar_datos_graficas(
        '/content/Original/abril_2020',
        '/content/Generado/gen_abril_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Abril_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Abril_2020/0_Clusters_Original_Abril_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/0_Clusters_Generado_Abril_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/0_Comparativa_Abril_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/1_Clusters_Original_Abril_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/1_Clusters_Generado_Abril_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/1_Comparativa_Abril_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/2_Clusters_Original_Abril_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/2_Clusters_Generado_Abril_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/2_Comparativa_Abril_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Abril_2020/3_Clusters_Original_Abril_2020_Spot_Market_

In [45]:
# Diccionario con nombres e índices Mayo 2020
variables_mayo_2020 = {
    "Mayo 2020 PV": 0,
    "Mayo 2020 Wind": 1,
    "Mayo 2020 Consumption": 2,
    "Mayo 2020 Spot Market Price": 3,
    "Mayo 2020 Precip 1h Mm": 4,
    "Mayo 2020 Precip Type Idx": 5,
    "Mayo 2020 Prob Precip 1h P": 6,
    "Mayo 2020 Clear Sky Rad W": 7,
    "Mayo 2020 Clear Sky Energy 1h J": 8,
    "Mayo 2020 Diffuse Rad W": 9,
    "Mayo 2020 Diffuse Rad 1h Wh": 10,
    "Mayo 2020 Direct Rad W": 11,
    "Mayo 2020 Direct Rad 1h Wh": 12,
    "Mayo 2020 Global Rad W": 13,
    "Mayo 2020 Global Rad 1h Wh": 14,
    "Mayo 2020 Sunshine Duration 1h Min": 15,
    "Mayo 2020 Sun Azimuth D": 16,
    "Mayo 2020 Sun Elevation D": 17,
    "Mayo 2020 Low Cloud Cover P": 18,
    "Mayo 2020 Medium Cloud Cover P": 19,
    "Mayo 2020 High Cloud Cover P": 20,
    "Mayo 2020 Total Cloud Cover P": 21,
    "Mayo 2020 Effective Cloud Cover P": 22,
    "Mayo 2020 Temp": 23,
    "Mayo 2020 Relative Humidity 2m P": 24,
    "Mayo 2020 Dew Point 2m C": 25,
    "Mayo 2020 Wind Speed 2m Ms": 26,
    "Mayo 2020 Wind Dir 2m D": 27,
    "Mayo 2020 T 10m C": 28,
    "Mayo 2020 Relative Humidity 10m P": 29,
    "Mayo 2020 Dew Point 10m C": 30,
    "Mayo 2020 Wind Speed 10m Ms": 31,
    "Mayo 2020 Wind Dir 10m D": 32,
    "Mayo 2020 T 50m C": 33,
    "Mayo 2020 Relative Humidity 50m P": 34,
    "Mayo 2020 Dew Point 50m C": 35,
    "Mayo 2020 Wind Speed 50m Ms": 36,
    "Mayo 2020 Wind Dir 50m D": 37,
    "Mayo 2020 T 100m C": 38,
    "Mayo 2020 Relative Humidity 100m P": 39,
    "Mayo 2020 Dew Point 100m C": 40,
    "Mayo 2020 Wind Speed 100m Ms": 41,
    "Mayo 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_mayo_2020.items():
    guardar_datos_graficas(
        '/content/Original/mayo_2020',
        '/content/Generado/gen_mayo_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Mayo_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Mayo_2020/0_Clusters_Original_Mayo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/0_Clusters_Generado_Mayo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/0_Comparativa_Mayo_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/1_Clusters_Original_Mayo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/1_Clusters_Generado_Mayo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/1_Comparativa_Mayo_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/2_Clusters_Original_Mayo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/2_Clusters_Generado_Mayo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/2_Comparativa_Mayo_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Mayo_2020/3_Clusters_Original_Mayo_2020_Spot_Market_Price.png
Imagen gua

In [46]:
# Diccionario con nombres e índices Junio 2020
variables_junio_2020 = {
    "Junio 2020 PV": 0,
    "Junio 2020 Wind": 1,
    "Junio 2020 Consumption": 2,
    "Junio 2020 Spot Market Price": 3,
    "Junio 2020 Precip 1h Mm": 4,
    "Junio 2020 Precip Type Idx": 5,
    "Junio 2020 Prob Precip 1h P": 6,
    "Junio 2020 Clear Sky Rad W": 7,
    "Junio 2020 Clear Sky Energy 1h J": 8,
    "Junio 2020 Diffuse Rad W": 9,
    "Junio 2020 Diffuse Rad 1h Wh": 10,
    "Junio 2020 Direct Rad W": 11,
    "Junio 2020 Direct Rad 1h Wh": 12,
    "Junio 2020 Global Rad W": 13,
    "Junio 2020 Global Rad 1h Wh": 14,
    "Junio 2020 Sunshine Duration 1h Min": 15,
    "Junio 2020 Sun Azimuth D": 16,
    "Junio 2020 Sun Elevation D": 17,
    "Junio 2020 Low Cloud Cover P": 18,
    "Junio 2020 Medium Cloud Cover P": 19,
    "Junio 2020 High Cloud Cover P": 20,
    "Junio 2020 Total Cloud Cover P": 21,
    "Junio 2020 Effective Cloud Cover P": 22,
    "Junio 2020 Temp": 23,
    "Junio 2020 Relative Humidity 2m P": 24,
    "Junio 2020 Dew Point 2m C": 25,
    "Junio 2020 Wind Speed 2m Ms": 26,
    "Junio 2020 Wind Dir 2m D": 27,
    "Junio 2020 T 10m C": 28,
    "Junio 2020 Relative Humidity 10m P": 29,
    "Junio 2020 Dew Point 10m C": 30,
    "Junio 2020 Wind Speed 10m Ms": 31,
    "Junio 2020 Wind Dir 10m D": 32,
    "Junio 2020 T 50m C": 33,
    "Junio 2020 Relative Humidity 50m P": 34,
    "Junio 2020 Dew Point 50m C": 35,
    "Junio 2020 Wind Speed 50m Ms": 36,
    "Junio 2020 Wind Dir 50m D": 37,
    "Junio 2020 T 100m C": 38,
    "Junio 2020 Relative Humidity 100m P": 39,
    "Junio 2020 Dew Point 100m C": 40,
    "Junio 2020 Wind Speed 100m Ms": 41,
    "Junio 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_junio_2020.items():
    guardar_datos_graficas(
        '/content/Original/junio_2020',
        '/content/Generado/gen_junio_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Junio_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Junio_2020/0_Clusters_Original_Junio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/0_Clusters_Generado_Junio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/0_Comparativa_Junio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/1_Clusters_Original_Junio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/1_Clusters_Generado_Junio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/1_Comparativa_Junio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/2_Clusters_Original_Junio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/2_Clusters_Generado_Junio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/2_Comparativa_Junio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Junio_2020/3_Clusters_Original_Junio_2020_Spot_Market_

In [47]:
# Diccionario con nombres e índices Julio 2020
variables_julio_2020 = {
    "Julio 2020 PV": 0,
    "Julio 2020 Wind": 1,
    "Julio 2020 Consumption": 2,
    "Julio 2020 Spot Market Price": 3,
    "Julio 2020 Precip 1h Mm": 4,
    "Julio 2020 Precip Type Idx": 5,
    "Julio 2020 Prob Precip 1h P": 6,
    "Julio 2020 Clear Sky Rad W": 7,
    "Julio 2020 Clear Sky Energy 1h J": 8,
    "Julio 2020 Diffuse Rad W": 9,
    "Julio 2020 Diffuse Rad 1h Wh": 10,
    "Julio 2020 Direct Rad W": 11,
    "Julio 2020 Direct Rad 1h Wh": 12,
    "Julio 2020 Global Rad W": 13,
    "Julio 2020 Global Rad 1h Wh": 14,
    "Julio 2020 Sunshine Duration 1h Min": 15,
    "Julio 2020 Sun Azimuth D": 16,
    "Julio 2020 Sun Elevation D": 17,
    "Julio 2020 Low Cloud Cover P": 18,
    "Julio 2020 Medium Cloud Cover P": 19,
    "Julio 2020 High Cloud Cover P": 20,
    "Julio 2020 Total Cloud Cover P": 21,
    "Julio 2020 Effective Cloud Cover P": 22,
    "Julio 2020 Temp": 23,
    "Julio 2020 Relative Humidity 2m P": 24,
    "Julio 2020 Dew Point 2m C": 25,
    "Julio 2020 Wind Speed 2m Ms": 26,
    "Julio 2020 Wind Dir 2m D": 27,
    "Julio 2020 T 10m C": 28,
    "Julio 2020 Relative Humidity 10m P": 29,
    "Julio 2020 Dew Point 10m C": 30,
    "Julio 2020 Wind Speed 10m Ms": 31,
    "Julio 2020 Wind Dir 10m D": 32,
    "Julio 2020 T 50m C": 33,
    "Julio 2020 Relative Humidity 50m P": 34,
    "Julio 2020 Dew Point 50m C": 35,
    "Julio 2020 Wind Speed 50m Ms": 36,
    "Julio 2020 Wind Dir 50m D": 37,
    "Julio 2020 T 100m C": 38,
    "Julio 2020 Relative Humidity 100m P": 39,
    "Julio 2020 Dew Point 100m C": 40,
    "Julio 2020 Wind Speed 100m Ms": 41,
    "Julio 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_julio_2020.items():
    guardar_datos_graficas(
        '/content/Original/julio_2020',
        '/content/Generado/gen_julio_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Julio_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Julio_2020/0_Clusters_Original_Julio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/0_Clusters_Generado_Julio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/0_Comparativa_Julio_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/1_Clusters_Original_Julio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/1_Clusters_Generado_Julio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/1_Comparativa_Julio_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/2_Clusters_Original_Julio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/2_Clusters_Generado_Julio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/2_Comparativa_Julio_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Julio_2020/3_Clusters_Original_Julio_2020_Spot_Market_

In [48]:
# Diccionario con nombres e índices Agosto 2020
variables_agosto_2020 = {
    "Agosto 2020 PV": 0,
    "Agosto 2020 Wind": 1,
    "Agosto 2020 Consumption": 2,
    "Agosto 2020 Spot Market Price": 3,
    "Agosto 2020 Precip 1h Mm": 4,
    "Agosto 2020 Precip Type Idx": 5,
    "Agosto 2020 Prob Precip 1h P": 6,
    "Agosto 2020 Clear Sky Rad W": 7,
    "Agosto 2020 Clear Sky Energy 1h J": 8,
    "Agosto 2020 Diffuse Rad W": 9,
    "Agosto 2020 Diffuse Rad 1h Wh": 10,
    "Agosto 2020 Direct Rad W": 11,
    "Agosto 2020 Direct Rad 1h Wh": 12,
    "Agosto 2020 Global Rad W": 13,
    "Agosto 2020 Global Rad 1h Wh": 14,
    "Agosto 2020 Sunshine Duration 1h Min": 15,
    "Agosto 2020 Sun Azimuth D": 16,
    "Agosto 2020 Sun Elevation D": 17,
    "Agosto 2020 Low Cloud Cover P": 18,
    "Agosto 2020 Medium Cloud Cover P": 19,
    "Agosto 2020 High Cloud Cover P": 20,
    "Agosto 2020 Total Cloud Cover P": 21,
    "Agosto 2020 Effective Cloud Cover P": 22,
    "Agosto 2020 Temp": 23,
    "Agosto 2020 Relative Humidity 2m P": 24,
    "Agosto 2020 Dew Point 2m C": 25,
    "Agosto 2020 Wind Speed 2m Ms": 26,
    "Agosto 2020 Wind Dir 2m D": 27,
    "Agosto 2020 T 10m C": 28,
    "Agosto 2020 Relative Humidity 10m P": 29,
    "Agosto 2020 Dew Point 10m C": 30,
    "Agosto 2020 Wind Speed 10m Ms": 31,
    "Agosto 2020 Wind Dir 10m D": 32,
    "Agosto 2020 T 50m C": 33,
    "Agosto 2020 Relative Humidity 50m P": 34,
    "Agosto 2020 Dew Point 50m C": 35,
    "Agosto 2020 Wind Speed 50m Ms": 36,
    "Agosto 2020 Wind Dir 50m D": 37,
    "Agosto 2020 T 100m C": 38,
    "Agosto 2020 Relative Humidity 100m P": 39,
    "Agosto 2020 Dew Point 100m C": 40,
    "Agosto 2020 Wind Speed 100m Ms": 41,
    "Agosto 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_agosto_2020.items():
    guardar_datos_graficas(
        '/content/Original/agosto_2020',
        '/content/Generado/gen_agosto_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Agosto_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Agosto_2020/0_Clusters_Original_Agosto_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/0_Clusters_Generado_Agosto_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/0_Comparativa_Agosto_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/1_Clusters_Original_Agosto_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/1_Clusters_Generado_Agosto_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/1_Comparativa_Agosto_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/2_Clusters_Original_Agosto_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/2_Clusters_Generado_Agosto_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/2_Comparativa_Agosto_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Agosto_2020/3_Clusters_Original_Agos

In [49]:
# Diccionario con nombres e índices Septiembre 2020
variables_septiembre_2020 = {
    "Septiembre 2020 PV": 0,
    "Septiembre 2020 Wind": 1,
    "Septiembre 2020 Consumption": 2,
    "Septiembre 2020 Spot Market Price": 3,
    "Septiembre 2020 Precip 1h Mm": 4,
    "Septiembre 2020 Precip Type Idx": 5,
    "Septiembre 2020 Prob Precip 1h P": 6,
    "Septiembre 2020 Clear Sky Rad W": 7,
    "Septiembre 2020 Clear Sky Energy 1h J": 8,
    "Septiembre 2020 Diffuse Rad W": 9,
    "Septiembre 2020 Diffuse Rad 1h Wh": 10,
    "Septiembre 2020 Direct Rad W": 11,
    "Septiembre 2020 Direct Rad 1h Wh": 12,
    "Septiembre 2020 Global Rad W": 13,
    "Septiembre 2020 Global Rad 1h Wh": 14,
    "Septiembre 2020 Sunshine Duration 1h Min": 15,
    "Septiembre 2020 Sun Azimuth D": 16,
    "Septiembre 2020 Sun Elevation D": 17,
    "Septiembre 2020 Low Cloud Cover P": 18,
    "Septiembre 2020 Medium Cloud Cover P": 19,
    "Septiembre 2020 High Cloud Cover P": 20,
    "Septiembre 2020 Total Cloud Cover P": 21,
    "Septiembre 2020 Effective Cloud Cover P": 22,
    "Septiembre 2020 Temp": 23,
    "Septiembre 2020 Relative Humidity 2m P": 24,
    "Septiembre 2020 Dew Point 2m C": 25,
    "Septiembre 2020 Wind Speed 2m Ms": 26,
    "Septiembre 2020 Wind Dir 2m D": 27,
    "Septiembre 2020 T 10m C": 28,
    "Septiembre 2020 Relative Humidity 10m P": 29,
    "Septiembre 2020 Dew Point 10m C": 30,
    "Septiembre 2020 Wind Speed 10m Ms": 31,
    "Septiembre 2020 Wind Dir 10m D": 32,
    "Septiembre 2020 T 50m C": 33,
    "Septiembre 2020 Relative Humidity 50m P": 34,
    "Septiembre 2020 Dew Point 50m C": 35,
    "Septiembre 2020 Wind Speed 50m Ms": 36,
    "Septiembre 2020 Wind Dir 50m D": 37,
    "Septiembre 2020 T 100m C": 38,
    "Septiembre 2020 Relative Humidity 100m P": 39,
    "Septiembre 2020 Dew Point 100m C": 40,
    "Septiembre 2020 Wind Speed 100m Ms": 41,
    "Septiembre 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_septiembre_2020.items():
    guardar_datos_graficas(
        '/content/Original/septiembre_2020',
        '/content/Generado/gen_septiembre_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Septiembre_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/0_Clusters_Original_Septiembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/0_Clusters_Generado_Septiembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/0_Comparativa_Septiembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/1_Clusters_Original_Septiembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/1_Clusters_Generado_Septiembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/1_Comparativa_Septiembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/2_Clusters_Original_Septiembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/2_Clusters_Generado_Septiembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Septiembre_2020/2_Comparativa_Septiembre_2020_Consumption.png
Imagen guarda

In [50]:
# Diccionario con nombres e índices Octubre 2020
variables_octubre_2020 = {
    "Octubre 2020 PV": 0,
    "Octubre 2020 Wind": 1,
    "Octubre 2020 Consumption": 2,
    "Octubre 2020 Spot Market Price": 3,
    "Octubre 2020 Precip 1h Mm": 4,
    "Octubre 2020 Precip Type Idx": 5,
    "Octubre 2020 Prob Precip 1h P": 6,
    "Octubre 2020 Clear Sky Rad W": 7,
    "Octubre 2020 Clear Sky Energy 1h J": 8,
    "Octubre 2020 Diffuse Rad W": 9,
    "Octubre 2020 Diffuse Rad 1h Wh": 10,
    "Octubre 2020 Direct Rad W": 11,
    "Octubre 2020 Direct Rad 1h Wh": 12,
    "Octubre 2020 Global Rad W": 13,
    "Octubre 2020 Global Rad 1h Wh": 14,
    "Octubre 2020 Sunshine Duration 1h Min": 15,
    "Octubre 2020 Sun Azimuth D": 16,
    "Octubre 2020 Sun Elevation D": 17,
    "Octubre 2020 Low Cloud Cover P": 18,
    "Octubre 2020 Medium Cloud Cover P": 19,
    "Octubre 2020 High Cloud Cover P": 20,
    "Octubre 2020 Total Cloud Cover P": 21,
    "Octubre 2020 Effective Cloud Cover P": 22,
    "Octubre 2020 Temp": 23,
    "Octubre 2020 Relative Humidity 2m P": 24,
    "Octubre 2020 Dew Point 2m C": 25,
    "Octubre 2020 Wind Speed 2m Ms": 26,
    "Octubre 2020 Wind Dir 2m D": 27,
    "Octubre 2020 T 10m C": 28,
    "Octubre 2020 Relative Humidity 10m P": 29,
    "Octubre 2020 Dew Point 10m C": 30,
    "Octubre 2020 Wind Speed 10m Ms": 31,
    "Octubre 2020 Wind Dir 10m D": 32,
    "Octubre 2020 T 50m C": 33,
    "Octubre 2020 Relative Humidity 50m P": 34,
    "Octubre 2020 Dew Point 50m C": 35,
    "Octubre 2020 Wind Speed 50m Ms": 36,
    "Octubre 2020 Wind Dir 50m D": 37,
    "Octubre 2020 T 100m C": 38,
    "Octubre 2020 Relative Humidity 100m P": 39,
    "Octubre 2020 Dew Point 100m C": 40,
    "Octubre 2020 Wind Speed 100m Ms": 41,
    "Octubre 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_octubre_2020.items():
    guardar_datos_graficas(
        '/content/Original/octubre_2020',
        '/content/Generado/gen_octubre_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Octubre_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Octubre_2020/0_Clusters_Original_Octubre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/0_Clusters_Generado_Octubre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/0_Comparativa_Octubre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/1_Clusters_Original_Octubre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/1_Clusters_Generado_Octubre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/1_Comparativa_Octubre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/2_Clusters_Original_Octubre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/2_Clusters_Generado_Octubre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/2_Comparativa_Octubre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Octubre_2020/3_Clu

In [51]:
# Diccionario con nombres e índices Noviembre 2020
variables_noviembre_2020 = {
    "Noviembre 2020 PV": 0,
    "Noviembre 2020 Wind": 1,
    "Noviembre 2020 Consumption": 2,
    "Noviembre 2020 Spot Market Price": 3,
    "Noviembre 2020 Precip 1h Mm": 4,
    "Noviembre 2020 Precip Type Idx": 5,
    "Noviembre 2020 Prob Precip 1h P": 6,
    "Noviembre 2020 Clear Sky Rad W": 7,
    "Noviembre 2020 Clear Sky Energy 1h J": 8,
    "Noviembre 2020 Diffuse Rad W": 9,
    "Noviembre 2020 Diffuse Rad 1h Wh": 10,
    "Noviembre 2020 Direct Rad W": 11,
    "Noviembre 2020 Direct Rad 1h Wh": 12,
    "Noviembre 2020 Global Rad W": 13,
    "Noviembre 2020 Global Rad 1h Wh": 14,
    "Noviembre 2020 Sunshine Duration 1h Min": 15,
    "Noviembre 2020 Sun Azimuth D": 16,
    "Noviembre 2020 Sun Elevation D": 17,
    "Noviembre 2020 Low Cloud Cover P": 18,
    "Noviembre 2020 Medium Cloud Cover P": 19,
    "Noviembre 2020 High Cloud Cover P": 20,
    "Noviembre 2020 Total Cloud Cover P": 21,
    "Noviembre 2020 Effective Cloud Cover P": 22,
    "Noviembre 2020 Temp": 23,
    "Noviembre 2020 Relative Humidity 2m P": 24,
    "Noviembre 2020 Dew Point 2m C": 25,
    "Noviembre 2020 Wind Speed 2m Ms": 26,
    "Noviembre 2020 Wind Dir 2m D": 27,
    "Noviembre 2020 T 10m C": 28,
    "Noviembre 2020 Relative Humidity 10m P": 29,
    "Noviembre 2020 Dew Point 10m C": 30,
    "Noviembre 2020 Wind Speed 10m Ms": 31,
    "Noviembre 2020 Wind Dir 10m D": 32,
    "Noviembre 2020 T 50m C": 33,
    "Noviembre 2020 Relative Humidity 50m P": 34,
    "Noviembre 2020 Dew Point 50m C": 35,
    "Noviembre 2020 Wind Speed 50m Ms": 36,
    "Noviembre 2020 Wind Dir 50m D": 37,
    "Noviembre 2020 T 100m C": 38,
    "Noviembre 2020 Relative Humidity 100m P": 39,
    "Noviembre 2020 Dew Point 100m C": 40,
    "Noviembre 2020 Wind Speed 100m Ms": 41,
    "Noviembre 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_noviembre_2020.items():
    guardar_datos_graficas(
        '/content/Original/noviembre_2020',
        '/content/Generado/gen_noviembre_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Noviembre_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/0_Clusters_Original_Noviembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/0_Clusters_Generado_Noviembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/0_Comparativa_Noviembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/1_Clusters_Original_Noviembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/1_Clusters_Generado_Noviembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/1_Comparativa_Noviembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/2_Clusters_Original_Noviembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/2_Clusters_Generado_Noviembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Noviembre_2020/2_Comparativa_Noviembre_2020_Consumption.png
Imagen guardada en: /content/Gr

In [52]:
# Diccionario con nombres e índices Diciembre 2020
variables_diciembre_2020 = {
    "Diciembre 2020 PV": 0,
    "Diciembre 2020 Wind": 1,
    "Diciembre 2020 Consumption": 2,
    "Diciembre 2020 Spot Market Price": 3,
    "Diciembre 2020 Precip 1h Mm": 4,
    "Diciembre 2020 Precip Type Idx": 5,
    "Diciembre 2020 Prob Precip 1h P": 6,
    "Diciembre 2020 Clear Sky Rad W": 7,
    "Diciembre 2020 Clear Sky Energy 1h J": 8,
    "Diciembre 2020 Diffuse Rad W": 9,
    "Diciembre 2020 Diffuse Rad 1h Wh": 10,
    "Diciembre 2020 Direct Rad W": 11,
    "Diciembre 2020 Direct Rad 1h Wh": 12,
    "Diciembre 2020 Global Rad W": 13,
    "Diciembre 2020 Global Rad 1h Wh": 14,
    "Diciembre 2020 Sunshine Duration 1h Min": 15,
    "Diciembre 2020 Sun Azimuth D": 16,
    "Diciembre 2020 Sun Elevation D": 17,
    "Diciembre 2020 Low Cloud Cover P": 18,
    "Diciembre 2020 Medium Cloud Cover P": 19,
    "Diciembre 2020 High Cloud Cover P": 20,
    "Diciembre 2020 Total Cloud Cover P": 21,
    "Diciembre 2020 Effective Cloud Cover P": 22,
    "Diciembre 2020 Temp": 23,
    "Diciembre 2020 Relative Humidity 2m P": 24,
    "Diciembre 2020 Dew Point 2m C": 25,
    "Diciembre 2020 Wind Speed 2m Ms": 26,
    "Diciembre 2020 Wind Dir 2m D": 27,
    "Diciembre 2020 T 10m C": 28,
    "Diciembre 2020 Relative Humidity 10m P": 29,
    "Diciembre 2020 Dew Point 10m C": 30,
    "Diciembre 2020 Wind Speed 10m Ms": 31,
    "Diciembre 2020 Wind Dir 10m D": 32,
    "Diciembre 2020 T 50m C": 33,
    "Diciembre 2020 Relative Humidity 50m P": 34,
    "Diciembre 2020 Dew Point 50m C": 35,
    "Diciembre 2020 Wind Speed 50m Ms": 36,
    "Diciembre 2020 Wind Dir 50m D": 37,
    "Diciembre 2020 T 100m C": 38,
    "Diciembre 2020 Relative Humidity 100m P": 39,
    "Diciembre 2020 Dew Point 100m C": 40,
    "Diciembre 2020 Wind Speed 100m Ms": 41,
    "Diciembre 2020 Wind Dir 100m D": 42
}
# Loop sobre el diccionario
for nombre, idx in variables_diciembre_2020.items():
    guardar_datos_graficas(
        '/content/Original/diciembre_2020',
        '/content/Generado/gen_diciembre_2020',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Diciembre_2020'
    )

Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/0_Clusters_Original_Diciembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/0_Clusters_Generado_Diciembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/0_Comparativa_Diciembre_2020_PV.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/1_Clusters_Original_Diciembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/1_Clusters_Generado_Diciembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/1_Comparativa_Diciembre_2020_Wind.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/2_Clusters_Original_Diciembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/2_Clusters_Generado_Diciembre_2020_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Diciembre_2020/2_Comparativa_Diciembre_2020_Consumption.png
Imagen guardada en: /content/Gr

In [53]:
# Diccionario con nombres e índices Enero 2021
variables_enero_2021 = {
    "Enero 2021 PV": 0,
    "Enero 2021 Wind": 1,
    "Enero 2021 Consumption": 2,
    "Enero 2021 Spot Market Price": 3,
    "Enero 2021 Precip 1h Mm": 4,
    "Enero 2021 Precip Type Idx": 5,
    "Enero 2021 Prob Precip 1h P": 6,
    "Enero 2021 Clear Sky Rad W": 7,
    "Enero 2021 Clear Sky Energy 1h J": 8,
    "Enero 2021 Diffuse Rad W": 9,
    "Enero 2021 Diffuse Rad 1h Wh": 10,
    "Enero 2021 Direct Rad W": 11,
    "Enero 2021 Direct Rad 1h Wh": 12,
    "Enero 2021 Global Rad W": 13,
    "Enero 2021 Global Rad 1h Wh": 14,
    "Enero 2021 Sunshine Duration 1h Min": 15,
    "Enero 2021 Sun Azimuth D": 16,
    "Enero 2021 Sun Elevation D": 17,
    "Enero 2021 Low Cloud Cover P": 18,
    "Enero 2021 Medium Cloud Cover P": 19,
    "Enero 2021 High Cloud Cover P": 20,
    "Enero 2021 Total Cloud Cover P": 21,
    "Enero 2021 Effective Cloud Cover P": 22,
    "Enero 2021 Temp": 23,
    "Enero 2021 Relative Humidity 2m P": 24,
    "Enero 2021 Dew Point 2m C": 25,
    "Enero 2021 Wind Speed 2m Ms": 26,
    "Enero 2021 Wind Dir 2m D": 27,
    "Enero 2021 T 10m C": 28,
    "Enero 2021 Relative Humidity 10m P": 29,
    "Enero 2021 Dew Point 10m C": 30,
    "Enero 2021 Wind Speed 10m Ms": 31,
    "Enero 2021 Wind Dir 10m D": 32,
    "Enero 2021 T 50m C": 33,
    "Enero 2021 Relative Humidity 50m P": 34,
    "Enero 2021 Dew Point 50m C": 35,
    "Enero 2021 Wind Speed 50m Ms": 36,
    "Enero 2021 Wind Dir 50m D": 37,
    "Enero 2021 T 100m C": 38,
    "Enero 2021 Relative Humidity 100m P": 39,
    "Enero 2021 Dew Point 100m C": 40,
    "Enero 2021 Wind Speed 100m Ms": 41,
    "Enero 2021 Wind Dir 100m D": 42
}

# Loop sobre el diccionario
for nombre, idx in variables_enero_2021.items():
    guardar_datos_graficas(
        '/content/Original/enero_2021',
        '/content/Generado/gen_enero_2021',
        idx,
        1,
        f"{idx} Clusters Original {nombre}",
        f"{idx} Clusters Generado {nombre}",
        f"{idx} Comparativa {nombre}",
        '/content/Graficos_resultados/Enero_2021'
    )

Imagen guardada en: /content/Graficos_resultados/Enero_2021/0_Clusters_Original_Enero_2021_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/0_Clusters_Generado_Enero_2021_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/0_Comparativa_Enero_2021_PV.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/1_Clusters_Original_Enero_2021_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/1_Clusters_Generado_Enero_2021_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/1_Comparativa_Enero_2021_Wind.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/2_Clusters_Original_Enero_2021_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/2_Clusters_Generado_Enero_2021_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/2_Comparativa_Enero_2021_Consumption.png
Imagen guardada en: /content/Graficos_resultados/Enero_2021/3_Clusters_Original_Enero_2021_Spot_Market_

In [54]:
# Comprimir la carpeta "Categorias" en formato zip
shutil.make_archive("Resultados_de_graficas", 'zip', "/content/Graficos_resultados")

'/content/Resultados_de_graficas.zip'